In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os, sys, subprocess

print("TF", tf.__version__)

TF 2.21.0


This notebook walks a small model through the whole deployment path: train it, save it as a SavedModel, inspect it, convert it to TFLite, shrink it with quantization, and finish with a look at distributed training. The model is kept tiny on purpose so the focus stays on saving and optimizing, not on accuracy.

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print(x_train.shape, x_test.shape)

(60000, 28, 28) (10000, 28, 28)


A plain dense classifier. The first layer holds most of the weights, which is what makes the quantization size difference easy to see later.

In [3]:
def build_model():
    return keras.Sequential([
        keras.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10),
    ])

model = build_model()
model.compile(optimizer="adam",
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
base_loss, base_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Baseline test accuracy: {base_acc:.4f}")

Epoch 1/3
422/422 - 2s - 4ms/step - accuracy: 0.9181 - loss: 0.2869 - val_accuracy: 0.9662 - val_loss: 0.1199
Epoch 2/3
422/422 - 1s - 2ms/step - accuracy: 0.9687 - loss: 0.1060 - val_accuracy: 0.9700 - val_loss: 0.0999
Epoch 3/3
422/422 - 1s - 2ms/step - accuracy: 0.9785 - loss: 0.0708 - val_accuracy: 0.9792 - val_loss: 0.0729
Baseline test accuracy: 0.9759


A SavedModel is TensorFlow's standard, framework-level format for serving. In Keras 3, `model.export()` writes one. It is a directory, not a single file: `saved_model.pb` holds the graph and the serving signature, `variables/` holds the trained weights, `assets/` holds any extra files, and `fingerprint.pb` is a content hash.

In [5]:
SM_DIR = "mnist_savedmodel"
model.export(SM_DIR)
print("Contents:", sorted(os.listdir(SM_DIR)))

INFO:tensorflow:Assets written to: mnist_savedmodel/assets


INFO:tensorflow:Assets written to: mnist_savedmodel/assets


Saved artifact at 'mnist_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  5040298448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5040300176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5040299792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5040300560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5040299600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5040301328: TensorSpec(shape=(), dtype=tf.resource, name=None)
Contents: ['assets', 'fingerprint.pb', 'saved_model.pb', 'variables']


`saved_model_cli` is the built-in tool for inspecting a SavedModel without loading it in Python. The `serve` signature below is the exact input/output contract TF Serving would expose.

In [6]:
cli = os.path.join(os.path.dirname(sys.executable), "saved_model_cli")
out = subprocess.run([cli, "show", "--dir", SM_DIR, "--all"], capture_output=True, text=True)
print(out.stdout)


MetaGraphDef with tag-set: 'serve' contains the following SignatureDefs:

signature_def['__saved_model_init_op']:
  The given SavedModel SignatureDef contains the following input(s):
  The given SavedModel SignatureDef contains the following output(s):
    outputs['__saved_model_init_op'] tensor_info:
        dtype: DT_INVALID
        shape: unknown_rank
        name: NoOp
  Method name is: 

signature_def['serve']:
  The given SavedModel SignatureDef contains the following input(s):
    inputs['keras_tensor'] tensor_info:
        dtype: DT_FLOAT
        shape: (-1, 28, 28)
        name: serve_keras_tensor:0
  The given SavedModel SignatureDef contains the following output(s):
    outputs['output_0'] tensor_info:
        dtype: DT_FLOAT
        shape: (-1, 10)
        name: StatefulPartitionedCall:0
  Method name is: tensorflow/serving/predict

signature_def['serving_default']:
  The given SavedModel SignatureDef contains the following input(s):
    inputs['keras_tensor'] tensor_info:

TFLite is the format for phones and embedded devices. The converter reads the SavedModel and produces a single `.tflite` flatbuffer. This first conversion is the baseline, with no optimization applied.

In [7]:
converter = tf.lite.TFLiteConverter.from_saved_model(SM_DIR)
tflite_base = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_base)
print("Baseline TFLite bytes:", len(tflite_base))

Baseline TFLite bytes: 943584


W0000 00:00:1786743492.314434 27982551 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786743492.314456 27982551 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1786743492.315016 27982551 reader.cc:83] Reading SavedModel from: mnist_savedmodel
I0000 00:00:1786743492.315373 27982551 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1786743492.315378 27982551 reader.cc:147] Reading SavedModel debug info (if present) from: mnist_savedmodel
I0000 00:00:1786743492.317926 27982551 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1786743492.318258 27982551 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1786743492.330895 27982551 loader.cc:220] Running initialization op on SavedModel bundle at path: mnist_savedmodel
I0000 00:00:1786743492.334465 27982551 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 19451 microseconds.
I0000 00:00:1786743492.36841

Quantization shrinks the model by storing weights in fewer bits. Dynamic-range quantization converts the weights from 32-bit floats to 8-bit integers (about 4x smaller). Float16 quantization keeps floats but halves them to 16 bits. Both are post-training: no retraining, just a conversion setting.

In [8]:
# Dynamic-range quantization (int8 weights)
conv_dyn = tf.lite.TFLiteConverter.from_saved_model(SM_DIR)
conv_dyn.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dyn = conv_dyn.convert()
open("model_dynamic.tflite", "wb").write(tflite_dyn)

# Float16 quantization
conv_f16 = tf.lite.TFLiteConverter.from_saved_model(SM_DIR)
conv_f16.optimizations = [tf.lite.Optimize.DEFAULT]
conv_f16.target_spec.supported_types = [tf.float16]
tflite_f16 = conv_f16.convert()
open("model_float16.tflite", "wb").write(tflite_f16)

print("done")

W0000 00:00:1786743492.747396 27982551 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786743492.747411 27982551 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1786743492.747557 27982551 reader.cc:83] Reading SavedModel from: mnist_savedmodel
I0000 00:00:1786743492.747802 27982551 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1786743492.747806 27982551 reader.cc:147] Reading SavedModel debug info (if present) from: mnist_savedmodel
I0000 00:00:1786743492.749695 27982551 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1786743492.760273 27982551 loader.cc:220] Running initialization op on SavedModel bundle at path: mnist_savedmodel
I0000 00:00:1786743492.763897 27982551 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 16343 microseconds.


done


W0000 00:00:1786743493.045317 27982551 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786743493.045327 27982551 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1786743493.045474 27982551 reader.cc:83] Reading SavedModel from: mnist_savedmodel
I0000 00:00:1786743493.045709 27982551 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1786743493.045712 27982551 reader.cc:147] Reading SavedModel debug info (if present) from: mnist_savedmodel
I0000 00:00:1786743493.047481 27982551 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1786743493.063055 27982551 loader.cc:220] Running initialization op on SavedModel bundle at path: mnist_savedmodel
I0000 00:00:1786743493.066905 27982551 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 21434 microseconds.


In [9]:
def kb(n):
    return f"{n/1024:.1f} KB"

print("Baseline :", kb(len(tflite_base)))
print("Dynamic  :", kb(len(tflite_dyn)), f"({len(tflite_base)/len(tflite_dyn):.1f}x smaller)")
print("Float16  :", kb(len(tflite_f16)), f"({len(tflite_base)/len(tflite_f16):.1f}x smaller)")

Baseline : 921.5 KB
Dynamic  : 238.5 KB (3.9x smaller)
Float16  : 463.1 KB (2.0x smaller)


Smaller only matters if the model still works. We run the dynamic-range model through the TFLite interpreter on the test set and compare its accuracy to the original.

In [10]:
interpreter = tf.lite.Interpreter(model_content=tflite_dyn)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

preds = []
for img in x_test:
    interpreter.set_tensor(inp["index"], img[np.newaxis].astype("float32"))
    interpreter.invoke()
    preds.append(interpreter.get_tensor(out["index"])[0].argmax())

tflite_acc = (np.array(preds) == y_test).mean()
print(f"Baseline accuracy: {base_acc:.4f}")
print(f"Quantized accuracy: {tflite_acc:.4f}")

/Users/ducnguyen/Desktop/AIT506/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Baseline accuracy: 0.9759
Quantized accuracy: 0.9761


Distributed training spreads work across devices. `tf.distribute.MirroredStrategy` is the data-parallel one: it copies the model onto every GPU, gives each a slice of the batch, and averages the gradients each step. Building and compiling the model inside `strategy.scope()` is all it takes; the rest of the training code is unchanged. On this machine there is one device, so it runs as a single replica, but the same code scales to many.

In [11]:
strategy = tf.distribute.MirroredStrategy()
print("Replicas:", strategy.num_replicas_in_sync)

with strategy.scope():
    dist_model = build_model()
    dist_model.compile(optimizer="adam",
                       loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                       metrics=["accuracy"])

dist_model.fit(x_train, y_train, epochs=1, batch_size=128, verbose=2)
print("Distributed training ran under", type(strategy).__name__)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


Replicas: 1


W0000 00:00:1786743493.728281 27982551 dataset.cc:997] Input of GeneratorDatasetOp::Dataset will not be optimized because the dataset does not implement the AsGraphDefInternal() method needed to apply optimizations.


469/469 - 1s - 3ms/step - accuracy: 0.9225 - loss: 0.2693
Distributed training ran under MirroredStrategy
